# 04 — BiGRU + BPE + BIO baseline

**Pipeline:** original text → committed ByteLevel BPE → random trainable embeddings (128) → one bidirectional GRU (128 per direction) → dropout (0.2) → linear classifier (256 → 3) → BIO → original character spans → location strings → `comp_metric`.

Use the fixed validation manifest, all UK training rows, and its existing 100,000-row RU subset. No cleaning, resplitting, vocabulary fitting, pretrained embeddings, CRF, attention, or augmentation. Five epochs; no early stopping. Best model: UK competition F1.

This notebook has been prepared by static review only. **No runtime results are supplied.** Assignment/course PDFs were unavailable; no claims about their contents are made. `uk_testlike` is derived from validation and is not an independent holdout.



## 1. Environment and repository

Select a GPU runtime in Kaggle or Colab. In Kaggle, attach `kse-ua-location-extraction-2026`, enable Internet for setup, and configure the `GITHUB_TOKEN` secret if a clone is needed. In Colab, configure the same secret and authenticate Kaggle through its supported credentials after accepting the competition rules. Do not put credentials in notebook cells or outputs.

An existing clone may be supplied using `UA_NER_REPO_DIR`. Otherwise the cell clones the private repository's **baseline** branch into the environment's working directory. An existing directory must already contain this implementation; it is never reset or pulled automatically. No datasets are stored in Git. Execute cells in order, including dependency installation, before imports.



In [ ]:
import os
import sys
import subprocess
from pathlib import Path

ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"
WORK_DIR = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")
if not WORK_DIR.is_dir():
    raise RuntimeError("Run in Kaggle or Colab, or adapt the environment setup explicitly")
REPO_DIR = Path(os.environ.get("UA_NER_REPO_DIR", WORK_DIR / "UA-Locations-NER-Deep-Learning"))
if not REPO_DIR.exists():
    import base64
    if ENVIRONMENT == "colab":
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    else:
        from kaggle_secrets import UserSecretsClient
        github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError("Provide GITHUB_TOKEN as an environment secret or supply an existing clone")
    auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    clone_env = os.environ.copy()
    clone_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
                     GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone", "--branch", "baseline",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(REPO_DIR)], env=clone_env, check=True)
    finally:
        del github_token, auth, clone_env
assert (REPO_DIR / "src/model.py").is_file(), "Supply a clone containing the baseline implementation"
sys.path.insert(0, str(REPO_DIR))
print("Environment:", ENVIRONMENT, "Repository:", REPO_DIR)



In [ ]:
%pip install -q -r "$REPO_DIR/requirements.txt"



In [ ]:
import gc
import hashlib
import json
import platform
import random
import uuid
from datetime import datetime, timezone
from functools import partial
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
from tokenizers import Tokenizer
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.data_io import get_data_dir, load_manifest, load_data_splits
from src.eda_scan import parse_spans, has_pair_shape
from src.preprocessing import prepare_bio_data, bio_to_spans
from src.dataset import (NERDataset, LengthBatchSampler, collate_ner,
                         LABEL_TO_ID, ID_TO_LABEL, IGNORE_INDEX)
from src.model import BiGRUTagger
from src.metrics import comp_metric, location_errors



## 2. Configuration, reproducibility and output isolation

Full sequences are retained (`max_sequence_length=None`). The 16,384 padded-token budget changes batching only; a sequence exceeding that budget runs alone. This cannot guarantee GPU memory for an arbitrarily long single sequence. If memory is insufficient, lower batch size/token budget and restart the complete run; do not silently truncate or change the split.

The seed, actual dependency versions, source revision, tokenizer/manifest hashes and device are recorded. Deterministic PyTorch operations are requested; an unsupported operation raises rather than silently becoming nondeterministic. Exact reproducibility across different hardware/library versions is not guaranteed.



In [ ]:
CONFIG = {
    "seed": 17, "embedding_dim": 128, "hidden_size": 128, "dropout": 0.2,
    "num_layers": 1, "bidirectional": True, "batch_size": 128,
    "learning_rate": 1e-3, "optimizer": "Adam", "epochs": 5,
    "gradient_clip": 1.0, "max_sequence_length": None,
    "max_padded_tokens": 16384, "pool_size": 4096, "prepare_batch_size": 2000,
    "num_workers": 0, "sources": ["uk", "ru"],
    "tokenizer": "bpe_bytelevel.json", "selection_metric": "uk_comp_f1",
    "early_stopping": False, "label_to_id": LABEL_TO_ID,
    "ignore_index": IGNORE_INDEX, "environment": ENVIRONMENT,
}
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = WORK_DIR / "outputs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

def save_json(path, value):
    with Path(path).open("w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, allow_nan=False)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

SPLITS_DIR = REPO_DIR / "data/splits"
TOKENIZER_PATH = REPO_DIR / "artifacts/tokenizers" / CONFIG["tokenizer"]
revision = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
worktree_status = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "status", "--short"], text=True).strip()
PROVENANCE = {
    "run_id": RUN_ID, "seed": CONFIG["seed"], "git_revision": revision,
    "git_status": worktree_status, "python": platform.python_version(),
    "packages": {name: version(name) for name in ["numpy", "pandas", "pyarrow", "torch", "tokenizers", "tqdm"]},
    "device": str(DEVICE), "cuda_version": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "tokenizer_sha256": sha256_file(TOKENIZER_PATH),
    "manifest_sha256": sha256_file(SPLITS_DIR / "validation_manifest.parquet"),
    "implementation_sha256": {str(path.relative_to(REPO_DIR)): sha256_file(path)
                              for path in [*sorted((REPO_DIR / "src").glob("*.py")),
                                           REPO_DIR / "requirements.txt",
                                           REPO_DIR / "notebooks/04_baseline.ipynb"]},
    "diagnostic_sha256": {name: sha256_file(SPLITS_DIR / f"{name}.parquet")
                          for name in ["uk_testlike", "uk_testlike_original"]},
}
save_json(RUN_DIR / "config.json", CONFIG)
save_json(RUN_DIR / "provenance.json", PROVENANCE)
print("Device:", DEVICE, "Run:", RUN_DIR)
print("CPU execution of the full experiment can be slow.")



## 3. Fixed manifest and existing tokenizer

`load_data_splits` calls `get_data_dir`, `load_manifest` and `select_manifest_rows`. It returns DataFrames in RAM, not a streaming training dataset. Load one source at a time, retain only required columns, prepare in small batches and release each DataFrame. Token arrays use compact NumPy storage; training records discard character offsets after alignment. Validation retains original text and offsets for complete competition evaluation.

No original `is_valid` flags replace the committed manifest. UK grouping is by near-duplicate group; RU grouping is by `doc_id`. Group IDs are scoped by source.



In [ ]:
manifest = load_manifest(SPLITS_DIR)
assert not manifest.duplicated(["source", "row_id"]).any()
assert set(manifest["source"]) == {"uk", "ru"}
assert set(manifest["split"]) == {"train", "val"}
assert set(manifest["seed"]) == {CONFIG["seed"]}
assert not manifest["group_id"].isna().any()
for source in CONFIG["sources"]:
    part = manifest[manifest["source"] == source]
    train_part, val_part = part[part["split"] == "train"], part[part["split"] == "val"]
    assert set(train_part["row_id"]).isdisjoint(val_part["row_id"])
    assert set(train_part["group_id"]).isdisjoint(val_part["group_id"])
counts = pd.crosstab(manifest["source"], manifest["split"])
assert counts.loc["uk", "train"] == 805983 and counts.loc["uk", "val"] == 201498
assert counts.loc["ru", "train"] == 100000 and counts.loc["ru", "val"] == 49369
display(counts)
counts.to_csv(RUN_DIR / "manifest_counts.csv")

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tokenizer.no_padding()
tokenizer.no_truncation()
PAD_ID = tokenizer.token_to_id("[PAD]")
assert PAD_ID == 0 and tokenizer.token_to_id("[UNK]") == 1
assert tokenizer.get_vocab_size() == 32000
DATA_DIR = get_data_dir(ENVIRONMENT)
print("Data directory:", DATA_DIR)
print("Tokenizer vocabulary:", tokenizer.get_vocab_size())



## 4. BIO preparation and coverage audit

Reuse `prepare_bio_data` without changing its exact span-boundary policy. Each preparation batch is validated structurally first. Unexpected malformed/overlapping gold annotations stop the run. Alignment exclusions are logged by original source and row ID, never converted to fabricated O labels.

Training excludes unaligned or zero-token examples. **Every original validation row remains in inference/scoring**, including alignment failures; only its supervised loss is unavailable. Gold competition strings always come from original annotations. Empty texts receive a masked dummy token for packing and produce no spans. ByteLevel may give repeated/overlapping character offsets for Unicode bytes; these are preserved.

The saved preprocessing notebook reports ByteLevel training skips UK=1,602/RU=110 and validation skips UK=361/RU=44. These are reference diagnostics, not results from this run; actual counts are measured below. Round-trip checks apply to every aligned example.



In [ ]:
ALIGNMENT_STATS = []
AUDIT_PATH = RUN_DIR / "alignment_exclusions.jsonl"

def checked_spans(text, raw):
    spans = parse_spans(raw)
    if not isinstance(text, str) or not has_pair_shape(spans):
        raise ValueError("Malformed text/annotation: inspect original source row")
    if not all(0 <= start < end <= len(text) for start, end in spans):
        raise ValueError("Out-of-bounds gold span")
    ordered = sorted(spans)
    if any(right[0] < left[1] for left, right in zip(ordered, ordered[1:])):
        raise ValueError("Overlapping gold spans cannot be represented by this BIO baseline")
    return spans


def prepare_frame(df, source, split, keep_for_evaluation):
    records = []
    stats = {"source": source, "split": split, "total": len(df),
             "aligned": 0, "alignment_failures": 0, "empty_token_sequences": 0,
             "supervised": 0, "retained": 0}
    with AUDIT_PATH.open("a", encoding="utf-8") as audit:
        for start in tqdm(range(0, len(df), CONFIG["prepare_batch_size"]),
                          desc=f"Prepare {source}/{split}"):
            batch = df.iloc[start:start + CONFIG["prepare_batch_size"]]
            gold_by_id = {}
            for row_id, row in batch.iterrows():
                gold_by_id[f"{source}_{row_id}"] = checked_spans(row["text"], row["loc_markers"])
            prepared, skipped = prepare_bio_data(tokenizer, batch, source)
            by_id = {record["text_id"]: record for record in prepared}
            skipped = set(skipped)
            assert len(by_id) + len(skipped) == len(batch)
            for row_id, row in batch.iterrows():
                text_id = f"{source}_{row_id}"
                original = by_id.get(text_id)
                spans = gold_by_id[text_id]
                if original is None:
                    assert text_id in skipped
                    stats["alignment_failures"] += 1
                    audit.write(json.dumps({"text_id": text_id, "source": source,
                                            "row_id": int(row_id), "split": split,
                                            "reason": "span_token_alignment"}) + "\n")
                    if not keep_for_evaluation:
                        continue
                    encoding = tokenizer.encode(row["text"], add_special_tokens=False)
                    ids, offsets, labels = encoding.ids, encoding.offsets, None
                else:
                    stats["aligned"] += 1
                    ids, offsets = original["input_ids"], original["offsets"]
                    assert sorted(bio_to_spans(original["labels"], offsets)) == sorted(spans)
                    labels = np.asarray([LABEL_TO_ID[value] for value in original["labels"]], dtype=np.int8)
                assert len(ids) == len(offsets)
                assert all(0 <= a <= b <= len(row["text"]) for a, b in offsets)
                if not ids:
                    stats["empty_token_sequences"] += 1
                    audit.write(json.dumps({"text_id": text_id, "source": source,
                                            "row_id": int(row_id), "split": split,
                                            "reason": "empty_token_sequence"}) + "\n")
                    if not keep_for_evaluation:
                        continue
                    labels = None
                if labels is not None:
                    stats["supervised"] += 1
                record = {"text_id": text_id, "source": source, "row_id": int(row_id),
                          "input_ids": np.asarray(ids, dtype=np.int32), "labels": labels}
                if keep_for_evaluation:
                    record.update(text=row["text"], offsets=np.asarray(offsets, dtype=np.int32).reshape(-1, 2),
                                  gold_spans=spans,
                                  gold_locations=[row["text"][a:b] for a, b in spans])
                records.append(record)
    stats["retained"] = len(records)
    if keep_for_evaluation:
        assert len(records) == len(df)
    ALIGNMENT_STATS.append(stats)
    return records

train_records, val_records = [], []
for source in CONFIG["sources"]:
    frames = load_data_splits(ENVIRONMENT, SPLITS_DIR, sources=(source,))
    for split in ["train", "val"]:
        frame = frames.pop(f"{source}_{split}")[["text", "loc_markers"]]
        expected = manifest[(manifest["source"] == source) & (manifest["split"] == split)]["row_id"]
        assert frame.index.tolist() == expected.tolist()
        prepared_records = prepare_frame(frame, source, split, keep_for_evaluation=(split == "val"))
        (train_records if split == "train" else val_records).extend(prepared_records)
        del frame, prepared_records
        gc.collect()
    del frames
assert len(val_records) == int(counts["val"].sum())
assert len({record["text_id"] for record in val_records}) == len(val_records)
alignment_table = pd.DataFrame(ALIGNMENT_STATS)
display(alignment_table)
alignment_table.to_csv(RUN_DIR / "alignment_counts.csv", index=False)
del manifest
gc.collect()



## 5. Dataset and dynamic batches

Training uses shuffled pools sorted by length, then shuffles the resulting batches. Every retained record appears once per epoch. Validation batches can be reordered for efficiency: results are always keyed by original `text_id`. The collator creates padding masks and ignores unavailable/padded labels with `-100`. Packing prevents both GRU directions from learning from padding.



In [ ]:
train_dataset, val_dataset = NERDataset(train_records), NERDataset(val_records)
train_sampler = LengthBatchSampler(train_dataset.lengths, CONFIG["batch_size"], shuffle=True,
                                  seed=CONFIG["seed"], max_padded_tokens=CONFIG["max_padded_tokens"],
                                  pool_size=CONFIG["pool_size"])
val_sampler = LengthBatchSampler(val_dataset.lengths, CONFIG["batch_size"],
                                max_padded_tokens=CONFIG["max_padded_tokens"])
loader_options = {"collate_fn": partial(collate_ner, pad_id=PAD_ID),
                  "num_workers": CONFIG["num_workers"], "pin_memory": DEVICE.type == "cuda"}
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, **loader_options)
val_loader = DataLoader(val_dataset, batch_sampler=val_sampler, **loader_options)
for name, dataset in [("train", train_dataset), ("val", val_dataset)]:
    lengths = np.asarray(dataset.lengths)
    print(name, "rows:", len(dataset), "length percentiles:", np.percentile(lengths, [50, 95, 99, 100]))
    print("Single sequences exceeding batching token budget:", int((lengths > CONFIG["max_padded_tokens"]).sum()))
    array_bytes = sum(value.nbytes for record in dataset.records for value in record.values()
                      if isinstance(value, np.ndarray))
    print("Compact array storage (GiB; excludes Python/text/loader overhead):", round(array_bytes / 1024**3, 3))



## 6. Model, loss and optimizer

The GRU has exactly **one stacked recurrent layer**, with two directions. Input `[B,T]` becomes embeddings `[B,T,128]`, packed GRU outputs `[B,T,256]`, and logits `[B,T,3]`. `O=0`, `B-LOC=1`, `I-LOC=2`; padding is not a fourth class. Random embeddings are trainable, with the padding row fixed by `padding_idx`.

Use unweighted token cross-entropy. Summed loss is divided by the number of supervised tokens for each update and for epoch reporting. The validation denominator excludes unavailable gold BIO labels, while competition F1 includes all original rows.



In [ ]:
model = BiGRUTagger(tokenizer.get_vocab_size(), pad_id=PAD_ID,
                    embedding_dim=CONFIG["embedding_dim"], hidden_size=CONFIG["hidden_size"],
                    dropout=CONFIG["dropout"], num_labels=len(LABEL_TO_ID)).to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX, reduction="sum")
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
print(model)
print("Trainable parameters:", sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))



## 7. Training and complete validation evaluation

The official metric in `src/metrics.py` is copied with the supplied literal first regex (no f-string), empty-string set members, empty-set rules and eight-decimal rounding intact. It must not be replaced with a conventional punctuation cleaner or token F1.

Argmax labels are decoded using the existing `bio_to_spans`; an orphan `I-LOC` starts a span, matching that function's established behavior. Extract locations using original `[start,end)` positions. A zero-width decoded span contains no text and is omitted. All other predictions, including duplicate strings, are passed to the metric, which performs its own normalization and set conversion.



In [ ]:
def train_epoch(epoch):
    model.train()
    train_sampler.set_epoch(epoch)
    total_loss, total_tokens = 0.0, 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch} train")
    for batch in progress:
        ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        targets = batch["labels"].to(DEVICE, non_blocking=True)
        supervised = int(targets.ne(IGNORE_INDEX).sum().item())
        if not supervised:
            continue
        optimizer.zero_grad(set_to_none=True)
        logits = model(ids, batch["lengths"])
        loss_sum = criterion(logits.reshape(-1, len(LABEL_TO_ID)), targets.reshape(-1))
        loss = loss_sum / supervised
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss")
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["gradient_clip"], error_if_nonfinite=True)
        optimizer.step()
        total_loss += loss_sum.item()
        total_tokens += supervised
        progress.set_postfix(loss=total_loss / total_tokens)
    if not total_tokens:
        raise RuntimeError("No supervised training tokens")
    return total_loss / total_tokens


@torch.no_grad()
def predict_loader(loader, description="Validation"):
    model.eval()
    predictions = {}
    loss_sums = {source: 0.0 for source in CONFIG["sources"]}
    token_counts = {source: 0 for source in CONFIG["sources"]}
    for batch in tqdm(loader, desc=description):
        ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        targets = batch["labels"].to(DEVICE, non_blocking=True)
        logits = model(ids, batch["lengths"])
        predicted_ids = logits.argmax(dim=-1).cpu().numpy()
        # Token losses remain separate so per-source means use correct denominators.
        losses = nn.functional.cross_entropy(logits.transpose(1, 2), targets,
                                             ignore_index=IGNORE_INDEX, reduction="none")
        row_losses = losses.sum(dim=1).cpu().tolist()
        row_tokens = targets.ne(IGNORE_INDEX).sum(dim=1).cpu().tolist()
        for index, record in enumerate(batch["records"]):
            source, text_id = record["source"], record["text_id"]
            if source in loss_sums:
                loss_sums[source] += row_losses[index]
                token_counts[source] += row_tokens[index]
            n = len(record["input_ids"])
            labels = [ID_TO_LABEL[int(value)] for value in predicted_ids[index, :n]]
            offsets = [tuple(map(int, pair)) for pair in record["offsets"]]
            spans = bio_to_spans(labels, offsets)
            assert all(0 <= a <= b <= len(record["text"]) for a, b in spans)
            spans = [(a, b) for a, b in spans if a < b]
            if text_id in predictions:
                raise ValueError(f"Duplicate prediction ID: {text_id}")
            predictions[text_id] = {"spans": spans, "locations": [record["text"][a:b] for a, b in spans]}
    assert len(predictions) == len(loader.dataset)
    losses = {source: loss_sums[source] / token_counts[source] if token_counts[source] else None
              for source in CONFIG["sources"]}
    total_tokens = sum(token_counts.values())
    losses["combined"] = sum(loss_sums.values()) / total_tokens if total_tokens else None
    return predictions, losses


def competition_scores(records, predictions):
    assert len(records) == len(predictions)
    scores = {}
    for source in [*CONFIG["sources"], "combined"]:
        selected = records if source == "combined" else [r for r in records if r["source"] == source]
        assert selected, f"No evaluation rows for {source}"
        scores[f"{source}_comp_f1"] = comp_metric(
            [r["gold_locations"] for r in selected],
            [predictions[r["text_id"]]["locations"] for r in selected])
    return scores


def save_predictions(path, records, predictions):
    # Key lookup makes output independent of sampler ordering.
    with Path(path).open("w", encoding="utf-8") as handle:
        for record in records:
            output = {"text_id": record["text_id"], "source": record["source"],
                      "row_id": record["row_id"], **predictions[record["text_id"]]}
            if "gold_locations" in record:
                output.update(gold_locations=record["gold_locations"],
                              **location_errors(record["gold_locations"], output["locations"]))
            handle.write(json.dumps(output, ensure_ascii=False) + "\n")



## 8. Five epochs and best checkpoint

No early stopping or scheduler. Validate after every epoch and retain the first checkpoint attaining the highest rounded UK competition F1. Each epoch writes ID-linked validation predictions and history. Atomic checkpoint replacement only updates this run's own `best.pt`.

The checkpoint includes optimizer and RNG states for provenance/recovery; this notebook starts a fresh run and does not automatically resume. Only load checkpoints created by this notebook from a trusted source.



In [ ]:
history = []
best_score, best_epoch = -1.0, None
BEST_PATH = RUN_DIR / "best.pt"
for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss = train_epoch(epoch)
    predictions, val_losses = predict_loader(val_loader, f"Epoch {epoch} validation")
    scores = competition_scores(val_records, predictions)
    row = {"epoch": epoch, "train_loss": train_loss,
           **{f"{source}_val_loss": value for source, value in val_losses.items()}, **scores}
    history.append(row)
    pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)
    save_predictions(RUN_DIR / f"val_predictions_epoch_{epoch}.jsonl", val_records, predictions)
    if scores["uk_comp_f1"] > best_score:
        best_score, best_epoch = scores["uk_comp_f1"], epoch
        checkpoint = {
            "epoch": epoch, "selection_score": best_score, "config": CONFIG,
            "sampler_epoch": epoch,
            "provenance": PROVENANCE, "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(), "history": list(history),
            "python_rng_state": random.getstate(), "numpy_rng_state": np.random.get_state(),
            "torch_rng_state": torch.get_rng_state(),
            "cuda_rng_state": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        }
        temporary_path = RUN_DIR / "best.tmp.pt"
        torch.save(checkpoint, temporary_path)
        temporary_path.replace(BEST_PATH)
        del checkpoint
    print(row, "Best epoch:", best_epoch)
    del predictions
    gc.collect()
display(pd.DataFrame(history))
# Free optimizer states and training records before final inference.
optimizer.zero_grad(set_to_none=True)
del optimizer, train_loader, train_sampler, train_dataset, train_records
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()



## 9. Reload the selected model and inspect errors

Recompute validation predictions from the saved best weights. Diagnostics include normalized FP/FN, exact character-span entity precision/recall/F1, negative-text false positives, per-source scores and token-length bins. Exact-span metrics count occurrences; competition F1 counts normalized strings once per text. These answer different questions.



In [ ]:
# weights_only=False is needed for this notebook's optimizer/NumPy RNG metadata.
# BEST_PATH was generated in this run; never substitute an untrusted checkpoint.
checkpoint = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
assert checkpoint["config"] == CONFIG
assert checkpoint["provenance"]["tokenizer_sha256"] == PROVENANCE["tokenizer_sha256"]
model.load_state_dict(checkpoint["model_state_dict"])
selected_epoch = checkpoint["epoch"]
del checkpoint
best_predictions, best_losses = predict_loader(val_loader, "Best checkpoint validation")
best_scores = competition_scores(val_records, best_predictions)
assert best_scores["uk_comp_f1"] == best_score, "Reloaded model changed the selection score"
save_predictions(RUN_DIR / "val_predictions_best.jsonl", val_records, best_predictions)


def diagnostic_summary(records, predictions):
    tp = fp = fn = negative = negative_fp = 0
    for record in records:
        gold, guess = set(record["gold_spans"]), set(predictions[record["text_id"]]["spans"])
        tp += len(gold & guess)
        fp += len(guess - gold)
        fn += len(gold - guess)
        if not gold:
            negative += 1
            negative_fp += bool(guess)
    precision = tp / (tp + fp) if tp + fp else (1.0 if not fn else 0.0)
    recall = tp / (tp + fn) if tp + fn else 1.0
    return {"rows": len(records), "span_precision": precision, "span_recall": recall,
            "span_f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
            "no_location_rows": negative, "no_location_rows_with_fp": negative_fp,
            "no_location_fp_rate": negative_fp / negative if negative else None}


diagnostics = []
for source in [*CONFIG["sources"], "combined"]:
    selected = val_records if source == "combined" else [r for r in val_records if r["source"] == source]
    diagnostics.append({"source": source, **diagnostic_summary(selected, best_predictions)})
display(pd.DataFrame(diagnostics))
pd.DataFrame(diagnostics).to_csv(RUN_DIR / "validation_diagnostics.csv", index=False)
length_diagnostics = []
for lower, upper in [(0, 32), (33, 64), (65, 128), (129, None)]:
    selected = [r for r in val_records if len(r["input_ids"]) >= lower
                and (upper is None or len(r["input_ids"]) <= upper)]
    if selected:
        length_diagnostics.append({"min_tokens": lower, "max_tokens": upper, "rows": len(selected),
            "comp_f1": comp_metric([r["gold_locations"] for r in selected],
                                   [best_predictions[r["text_id"]]["locations"] for r in selected])})
display(pd.DataFrame(length_diagnostics))
pd.DataFrame(length_diagnostics).to_csv(RUN_DIR / "length_diagnostics.csv", index=False)
shown = 0
for record in val_records:
    guess = best_predictions[record["text_id"]]
    errors = location_errors(record["gold_locations"], guess["locations"])
    if errors["false_positives"] or errors["false_negatives"]:
        print(record["text_id"], record["text"])
        print("Gold:", record["gold_locations"], "Predicted:", guess["locations"], errors)
        shown += 1
        if shown == 10:
            break
save_json(RUN_DIR / "summary.json", {"best_epoch": selected_epoch, **best_scores,
                                    "validation_losses": best_losses, "diagnostics": diagnostics})
print("Selected epoch:", selected_epoch, best_scores)
del best_predictions



## 10. Validation-derived domain-shift diagnostic

Compare the saved 5,000 original UK validation sentences with their 1,000 concatenated/noisy posts. These are **not independent holdouts** and are never training data or checkpoint-selection data. The number of texts and per-text deduplication change, so their score difference reflects both context/style changes and competition set aggregation.

Parquet annotations are arrays rather than CSV strings. Serialize a temporary copy before calling the existing parser; do not edit the saved artifacts. The source-row membership and exact extracted gold strings are checked before inference.



In [ ]:
original = pd.read_parquet(SPLITS_DIR / "uk_testlike_original.parquet")
testlike = pd.read_parquet(SPLITS_DIR / "uk_testlike.parquet")
assert len(original) == 5000 and len(testlike) == 1000
original_ids = set(map(int, original["row_id"]))
validation_uk_ids = {r["row_id"] for r in val_records if r["source"] == "uk"}
assert original_ids <= validation_uk_ids
used_ids = [int(value) for values in testlike["source_row_ids"] for value in values]
assert len(used_ids) == len(set(used_ids)) == len(original_ids)
assert set(used_ids) == original_ids


def serialize_parquet_spans(frame):
    result = frame[["text", "loc_markers"]].copy()
    result["loc_markers"] = result["loc_markers"].map(
        lambda spans: repr([tuple(map(int, span)) for span in spans]))
    return result


original_frame = serialize_parquet_spans(original)
original_frame.index = original["row_id"].to_numpy()
testlike_frame = serialize_parquet_spans(testlike)
original_diagnostic = prepare_frame(original_frame, "uk", "testlike_original", True)
testlike_diagnostic = prepare_frame(testlike_frame, "uk", "testlike", True)
original_gold = {r["row_id"]: r["gold_locations"] for r in original_diagnostic}
for record, source_ids in zip(testlike_diagnostic, testlike["source_row_ids"]):
    expected_gold = [location for row_id in source_ids for location in original_gold[int(row_id)]]
    assert record["gold_locations"] == expected_gold


def inference_loader(records):
    dataset = NERDataset(records)
    sampler = LengthBatchSampler(dataset.lengths, CONFIG["batch_size"],
                                 max_padded_tokens=CONFIG["max_padded_tokens"])
    return DataLoader(dataset, batch_sampler=sampler, **loader_options)


domain_scores = {}
for name, records in [("uk_testlike_original", original_diagnostic), ("uk_testlike", testlike_diagnostic)]:
    predictions, _ = predict_loader(inference_loader(records), name)
    domain_scores[name] = comp_metric([r["gold_locations"] for r in records],
                                     [predictions[r["text_id"]]["locations"] for r in records])
    save_predictions(RUN_DIR / f"{name}_predictions.jsonl", records, predictions)
    del predictions
save_json(RUN_DIR / "domain_shift_diagnostics.json", domain_scores)
pd.DataFrame(ALIGNMENT_STATS).to_csv(RUN_DIR / "alignment_counts.csv", index=False)
print("Validation-derived diagnostic scores:", domain_scores)
del original, testlike, original_frame, testlike_frame, original_diagnostic, testlike_diagnostic



## 11. Kaggle test inference and submission

Use the selected checkpoint, full original text and the committed tokenizer. `test.csv.locations` is a placeholder, never gold. Predictions are keyed by `text_id` despite length-sorted inference, then joined back in original test order. Serialize location lists with `json.dumps(ensure_ascii=False)` and let pandas handle CSV quoting. No Kaggle submission is sent automatically.



In [ ]:
test = pd.read_csv(DATA_DIR / "test.csv")
assert test["text_id"].is_unique and test["text_id"].notna().all()
assert test["text"].map(lambda value: isinstance(value, str)).all()
test_records = []
for row_id, row in test.iterrows():
    encoding = tokenizer.encode(row["text"], add_special_tokens=False)
    assert all(0 <= a <= b <= len(row["text"]) for a, b in encoding.offsets)
    test_records.append({"text_id": int(row["text_id"]), "row_id": int(row_id), "source": "test",
                         "text": row["text"], "input_ids": np.asarray(encoding.ids, dtype=np.int32),
                         "offsets": np.asarray(encoding.offsets, dtype=np.int32).reshape(-1, 2), "labels": None})
assert len({r["text_id"] for r in test_records}) == len(test)
test_predictions, _ = predict_loader(inference_loader(test_records), "Kaggle test")
assert set(test_predictions) == set(test["text_id"])
save_predictions(RUN_DIR / "test_predictions.jsonl", test_records, test_predictions)
submission = test[["text_id"]].copy()
submission["locations"] = submission["text_id"].map(
    lambda text_id: json.dumps(test_predictions[text_id]["locations"], ensure_ascii=False))
SUBMISSION_PATH = RUN_DIR / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
roundtrip = pd.read_csv(SUBMISSION_PATH)
assert roundtrip.columns.tolist() == ["text_id", "locations"]
assert roundtrip["text_id"].tolist() == test["text_id"].tolist()
assert [json.loads(value) for value in roundtrip["locations"]] == [
    test_predictions[text_id]["locations"] for text_id in test["text_id"]]
display(submission.head())
print("Saved submission:", SUBMISSION_PATH)



## 12. Results and export

Results are populated only by execution: `history.csv`, `summary.json`, `domain_shift_diagnostics.json`, ID-linked prediction JSONL files, alignment counts/exclusions, and `best.pt` belong to this unique run. No numerical model-performance claims are prefilled.

**First-run checks:** confirm manifest composition/group disjointness; tokenizer IDs/hash; alignment coverage and round trips; full validation row counts; finite losses; CPU/GPU memory; decreasing training loss; saved/reloaded best model; sensible original-text spans; and ID/JSON/CSV round trips for submission. Full runtime correctness and GPU capacity must be checked manually in your environment.

Kaggle: save a notebook version including outputs and download the run directory/archive from the Output panel. Colab: download the archive before the temporary runtime ends, using the Files panel or `files.download` below. Both the archive and checkpoint can be large. No Drive integration is required. Private-repository cloning works after this branch has been made available remotely; alternatively upload/provide a local clone containing these changes.



In [ ]:
import shutil
archive_path = shutil.make_archive(str(WORK_DIR / f"baseline_{RUN_ID}"), "zip", root_dir=RUN_DIR)
print("Export archive:", archive_path)
print("Run directory:", RUN_DIR)
# Colab: uncomment to trigger a browser download after inspecting the outputs.
# from google.colab import files
# files.download(archive_path)

